In [1]:
from py_mongosqltranslate import MongoSqlTranslator
from py_schema_builder_library import MongoConnectionProperties, SchemaBuilder
from pymongo import MongoClient


connection_properties = MongoConnectionProperties(uri="mongodb://localhost:27017/", username="testuser", password="password")
# Initialization and shape should be as close to the existing MongoClient as possible. We want the library to feel like it has the 
# same 'shape' as PyMongo
schema_builder = SchemaBuilder(connection_properties)
# TODO: The schema_builder should return the schema in the exact shape it needs to be passed to translate_query
movies_schema = schema_builder.sample_schema_for_collection(db_name="sample_mflix", collection_name="movies", sample_size=50)
# movies_schemas = schema_builder.build_schema(db_name="sample_mflix", collection_name="movies")
# print(movies_schema)
print(movies_schema)

# Note: Maybe we pass in a database here as a field. We don't support cross database queries, so maybe we make  it easier
# So the user doesn't need to specify the database each time.
# Likewise the user can add schemas or something as needed? And internally we check the catalog across runs?

"""
-- This is a loose idea for how support could look
-- Note that all the structs is different from the PyMongo client, so we should clean it up to make it as similar as possible
schema_builder = SchemaBuilder(connection_properties)
connection_properties = MongoConnectionProperties(uri="mongodb://localhost:27017/", username="testuser", password="password")
movies_schema = schema_builder.build_schema("sample_mflix", "movies")

translator = MongoSqlTranslator("sample_mflix")
translator.add_schema(database="sample_mflix", collection="movies", movies_schema)
translator.translate_query(sql="SELECT * FROM movies")
"""

translator = MongoSqlTranslator()
query_translation = translator.translate_query(current_db="sample_mflix", sql="SELECT title,runtime FROM movies WHERE runtime > 90 LIMIT 5",catalog= {
    "sample_mflix" : { "movies": movies_schema }
})

# Note: The API is really messy for generating a schema to passing it the translator, to finally executing via the Python Client
# For each query a user wants to run, they need to pass the database, and the collection, and a dictionary.
# Wondering if we could consolidate?

print(query_translation.pipeline)
# Might be worth working backwards here. Let's start with stubs for what our ideal interaction looks like application wise,
# And then clean it up to build out a nice api
client = MongoClient("mongodb://localhost:27017/")
db = client["sample_mflix"]
movies = db["movies"]
aggCursor = movies.aggregate(query_translation.pipeline)




{'$jsonSchema': {'bsonType': 'object', 'properties': {'tomatoes': {'bsonType': 'object', 'properties': {'consensus': {'bsonType': 'string'}, 'critic': {'bsonType': 'object', 'properties': {'meter': {'bsonType': 'int'}, 'numReviews': {'bsonType': 'int'}, 'rating': {'bsonType': 'double'}}, 'required': ['meter', 'numReviews', 'rating'], 'additionalProperties': False}, 'lastUpdated': {'bsonType': 'date'}, 'dvd': {'bsonType': 'date'}, 'rotten': {'bsonType': 'int'}, 'fresh': {'bsonType': 'int'}, 'boxOffice': {'bsonType': 'string'}, 'production': {'bsonType': 'string'}, 'viewer': {'bsonType': 'object', 'properties': {'meter': {'bsonType': 'int'}, 'numReviews': {'bsonType': 'int'}, 'rating': {'bsonType': 'double'}}, 'required': ['numReviews', 'rating'], 'additionalProperties': False}, 'website': {'bsonType': 'string'}}, 'required': ['lastUpdated', 'viewer'], 'additionalProperties': False}, 'fullplot': {'bsonType': 'string'}, 'genres': {'bsonType': 'array', 'items': {'bsonType': 'string'}}, 'me

In [2]:
# Question: is there a way for us to not have the single quote key  for every document? I want to avoid the replace with?
#{'': {'title': 'Dishonored', 'runtime': 91}}
# I think I need to update the PyMongoSQL translate to exclude namespaces?
for document in aggCursor:
   print(document)

{'': {'title': 'Dishonored', 'runtime': 91}}
{'': {'title': 'The Red Head', 'runtime': 91}}
{'': {'title': 'Dames', 'runtime': 91}}
{'': {'title': 'Twentieth Century', 'runtime': 91}}
{'': {'title': 'The Informer', 'runtime': 91}}
